# Verifying the author modified E-Flux algorithm bounds applied to the final data file (specifically for mouse cardiac simulation). 

This notebook assesses how the authors' modified E-Flux algorithm in mitoMammal (Chapman et al., 2025) converts mouse cardiac proteomic data into reaction flux bounds. For each complex of the electron transport chain (ETC) (Complexes I–V), the protein counts of every mouse subunit are normalised, combined according to the reaction's gene–protein–reaction (GPR) rule, and multiplied by the scaling constant of 50. The resulting values are compared with the bounds set by the authors' `Eflux()` function (`efflux_method.py`) before any optimisation (pFBA) is run.

## Testing both normalisations

In the published Methods (Section 2.5), the authors state that the data is scaled between 0 and 1 "by dividing by the 90th percentile", with values above 1 capped at 1.

However, in the cardiac notebook `proteomics_mouse_cardiac_vs_BAT.ipynb` downloaded from the authors' repository (https://gitlab.com/habermann_lab/mitomammal/), no quantile is specified. When `quantile` is not given, `read_csv_data()` divides by the maximum value. This notebook calculates the ETC bounds using both the 90th percentile and the maximum, to verify that the repository notebook does not scale to the 90th percentile unless this is specified.

## Code from the authors' `proteomics_mouse_cardiac_vs_BAT.ipynb` notebook (unchanged)

```python
def perform_FBA(file_path,id_gene,id_value,human,constant,objective_id):
    """This function open the model, constrain it with Omics data (human or not) and optimise the model with the selected objective function
    args:
        file_path (str):
        id_gene (int): position of the gene identifier in the dataframe
        id_value (int): position of the value in the dataframe
        human (bool): constrain data on humain or mouse data
        constant (int): constant value used to scale the constrain (only to make the value easier for human)
        objective_id (id): identifier of the objective function in the model
    return:
        model (cobra.core.model): a constrained model
        pfba (cobra.Solution.solution): solution of parsimonious FBA
    """
    model = cobra.io.read_sbml_model(join("6_universal_mito_model.xml")) 
    list_of_genes=[] 
    for gene in model.genes:
        list_of_genes.append(gene.id)
    data_dictionary = read_csv_data(path=file_path,
                                      list_of_genes=list_of_genes,
                                      id_gene=id_gene,
                                      id_val=id_value,
                                      sep="\t",
                                      head=True)
    #
    model.objective = model.reactions[objective_id] 
    Eflux(model,data_dictionary,constant,1, ignore_human= not human,ignore_mus=human)
    exch,const=0,0
    for react in model.reactions:
        if react.bounds[1] == constant:
            const+=1
        elif react.bounds[1] == 1000:
            exch+=1
    
    return [model,pfba(model)]
```

## Change applied to scale to the 90th percentile

To scale to the 90th percentile, as described in the published Methods, the `read_csv_data()` call above is changed by adding `quantile=0.9`:

```python
    data_dictionary = read_csv_data(path=file_path,
                                      list_of_genes=list_of_genes,
                                      id_gene=id_gene,
                                      id_val=id_value,
                                      sep="\t",
                                      head=True,
                                      quantile=0.9)   ### 90th percentile, as in the published Methods
```

## Code in `efflux_method.py` that specifies quantile)

The code below is copied from the authors' `efflux_method.py`:

- The function's `quantile` argument defaults to `np.nan` (not a number), meaning no quantile is set.
- `if quantile == quantile:` is only true when a number is specified.
- **If a quantile is given** : each value is divided by that percentile, and any value above 1 is capped at 1. This is the method described in the published Methods.
- **If no quantile is given** : each value is divided by the maximum value instead, and capping is therefore not necessary.

Only the genes that are in the model are used to calculate the percentile or maximum.

```python
def read_csv_data(path,id_gene,id_val,head,list_of_genes,sep=",",comment="_",quantile=np.nan): 
    ...
    dict_exp = {} 
    with open(path) as file: 
        for line in file:
            if head:
                head=False
                continue
            if line.startswith(comment): # If the line is a comment ignore it
                continue
            line = line.replace('"','').split(sep)
            if line[id_gene] in list_of_genes:
                dict_exp[line[id_gene]] = float(line[id_val]) # create the entry in the dictionnary with the gene id and its value
    if quantile == quantile:
        quantile = np.quantile(list(dict_exp.values()),quantile)
        for i in dict_exp: 
            dict_exp[i] = dict_exp[i]/quantile
            if dict_exp[i] > 1:
                dict_exp[i]=1
    else:
        maximum = max(dict_exp.values())
        for i in dict_exp: # Divide each entry by the maximum value
            dict_exp[i] = dict_exp[i]/maximum
    
    return dict_exp # return the dictionnary
```
### Details

**Model:** `6_universal_mito_model.xml` (SBML Level 3 Version 1), from the Habermann Lab GitLab repository (https://gitlab.com/habermann_lab/mitomammal/).

**Data:** `Heart_data.tsv`, mouse cardiac mitochondrial proteomic data (Hansen et al., 2024) supplied with mitoMammal. The file has no header row; column 0 is the mouse Ensembl gene ID and column 1 the protein count.

**Software:** Python 3.8.20 and


### Loading the model and the cardiac proteomic data

In [1]:
import cobra
import pandas as pd

model = cobra.io.read_sbml_model(r"C:\Users\cayle\OneDrive\Documents\MitoMammal Author Data\6_universal_mito_model.xml")  
data = pd.read_csv(r"C:\Users\cayle\OneDrive\Documents\MitoMammal Author Data\Heart_data.tsv",sep="\t", header=None, names=["gene ID", "value"])        

print(len(model.genes), "genes in model")
print(data.shape)
data.head()

782 genes in model
(6652, 2)


,gene ID,value
0,ENSMUSG00000031174,0
1,ENSMUSG00000031137,0
2,ENSMUSG00000094345,0
3,ENSMUSG00000096632,0
4,ENSMUSG00000093861,0


In [2]:
print(data.columns.tolist())
#confirming the assigned column names 

['gene ID', 'value']


### Genes Present in Both the Model and the Final Data File:

In [3]:
#Collecting the gene IDs from the model and from the data as sets
model_genes = {g.id for g in model.genes}
data_genes = set(data["gene ID"].dropna().astype(str))

In [4]:
#retaining only the genes found in both the model's GPR rules and the final data file

shared_mod_data = sorted(model_genes & data_genes)
print(len(shared_mod_data), "genes in both")

#Exporting the list as a csv file for interest sake
pd.DataFrame({"gene ID": shared_mod_data}).to_csv("genes_in_model_and_data.csv", index=False)

331 genes in both


### 90th Percentile & Maximum of the Shared Genes

These are the two options for normalisation of the protein counts: the 90th percentile value and the maximum value of the shared gene list.

In [5]:
subset_shared = data[data["gene ID"].astype(str).isin(shared_mod_data)]
prot_values = pd.to_numeric(subset_shared["value"], errors="coerce").dropna()

print(len(prot_values), "values used")
print("90th percentile:", prot_values.quantile(0.90))
print("Maximum:", prot_values.quantile(1.0))

331 values used
90th percentile: 163.0
Maximum: 367.0


In [6]:
#The p90 value is 163, and the maximum value is 367.0.
p90 = prot_values.quantile(0.90)

In [7]:
#Just looking which genes are in the top 10% (above 90th percentile)
#These genes would be capped at 1 when scaled between 0-1 by E-flux

subset_shared = subset_shared.assign(value=pd.to_numeric(subset_shared["value"], errors="coerce"))
top10 = subset_shared[subset_shared["value"] > prot_values.quantile(0.90)].sort_values("value", ascending=False)

print(len(top10), "genes above the 90th percentile")
top10

33 genes above the 90th percentile


,gene ID,value
3903,ENSMUSG00000020456,367
5522,ENSMUSG00000022477,360
83,ENSMUSG00000025968,305
4611,ENSMUSG00000025745,283
1376,ENSMUSG00000025428,268
1735,ENSMUSG00000024892,267
2847,ENSMUSG00000018796,260
2611,ENSMUSG00000037012,258
2992,ENSMUSG00000028607,257
4985,ENSMUSG00000021577,248


____________________________________________________________________________________

## Section 2
### Investigating the ETC Flux Bounds Using the 90th Percentile


The E-flux method has been delineated to work as follows: The protien abundance values are normalised, that number is multiplied by the GPR value, which is then multiplied by some constant (in mitoMammal that constant is 50), to obtain the flux bounds constrained by the data. The code below reports a summarises this information for each ETC complex for comparison. 

### E-Flux bounds from the authors' notebook (90th percentile)

To obtain the bounds set by E-Flux as a result of contextualisation with the cardiac proteomic data, the authors' cardiac notebook (`proteomics_mouse_cardiac_vs_BAT.ipynb`) was run, defining the `quantile=0.9` as detailed above. In the `proteomics_mouse_cardiac_vs_BAT.ipynb` notebook,  `perform_FBA()` returns the E-Flux-constrained model as `result_Heart[0]`. Running pFBA does not change the reaction bounds, so the bounds read from this model are those set by E-Flux algorithm. 

The bounds of the five ETC complexes were printed in the authors' notebook using:

```python
for rid in ["CI_mitoMap", "CII_mitoMap", "CIII_mitoMap", "CIV_mitoMap", "CV_mitoMap"]:
    print(rid, result_Heart[0].reactions.get_by_id(rid).bounds)
```

which gave the following (lower bound, upper bound):

```
CI_mitoMap   (-5.521472392638037, 5.521472392638037)
CII_mitoMap  (-10.736196319018406, 10.736196319018406)
CIII_mitoMap (-3.067484662576687, 3.067484662576687)
CIV_mitoMap  (0, 1.2269938650306749)
CV_mitoMap   (-0.9202453987730062, 0.9202453987730062)
```

The `summary()` cells below recalculate each of these bounds, so the "× 50" value printed for each complex should match the bounds above.

According to the following code run in the author's `proteomics_mouse_cardiac_vs_BAT.ipynb` notebook:

```python
model = result_Heart[0]
c=0
e=0
for reaction in model.reactions:
    c+= reaction._upper_bound != 50.0 and reaction._upper_bound != 1000.0
    e+= reaction._upper_bound == 1000.0
u = len(model.reactions) - c - e
print(f"There are :{c} constrained reactions\n           {u} unconstrained reactions\n           {e} exchange reactions")
```
There are 308 constrained reactions, 
           180 unconstrained reactions, 
           and 72 exchange reactions. This value differs from the number of constrained reactions reported in the publication, which is 330: "To demonstrate mitoMammal’s ability of modelling mouse cardiac mitochondrial metabolism, we integrated proteomic data harvested from mitochondria isolated from mouse cardiac tissue and optimized ATP hydrolysis. This resulted in 330 constrained reactions out of the complement of 560 reactions". 
           

In [8]:
import ast #this reads text written like Python code and breaks it into parts and its used here to read GPR rules
import re #re searches and splits text using patterns and is used here to exlclude the human genes from the GPR rule

#Joining the common names to their reaction IDs as written in the model. 
complexes = {
    "complex i":   "CI_mitoMap",
    "complex ii":  "CII_mitoMap",
    "complex iii": "CIII_mitoMap",
    "complex iv":  "CIV_mitoMap",
    "complex v":   "CV_mitoMap",
}

#lookup dictionary joining each gene ID to its protein count
vals = dict(zip(data["gene ID"].astype(str), pd.to_numeric(data["value"], errors="coerce")))

def gpr_value(node):
    if isinstance(node, ast.Expression):
        return gpr_value(node.body)
    # if single gene, return its protein count as is, divided by p90
    if isinstance(node, ast.Name):
        v = vals.get(node.id)
        return None if v is None or pd.isna(v) else v / p90
    # AND: for genes that encode subunits of a single complex all of which are required for the reaction, the minimum value is retained. 
    if isinstance(node, ast.BoolOp):
        parts = [p for p in (gpr_value(n) for n in node.values) if p is not None]
        if not parts:
            return None
        return min(parts) if isinstance(node.op, ast.And) else sum(parts)

#To print the summaries
def summary(name):
    rid = complexes[name.lower()]
    rxn = model.reactions.get_by_id(rid)

#retaining only the mouse GPR rule
    mouse_rule = re.split(r"\s+or\s+\(?ENSG\d", rxn.gene_reaction_rule)[0]
    mouse = [g.id for g in rxn.genes if g.id.startswith("ENSMUSG")]

    table = pd.DataFrame({"gene ID": mouse})
    table["value"] = table["gene ID"].map(vals)
    table["value / p90"] = table["value"] / p90

#combining the subunits according to their GPR rules
    result = gpr_value(ast.parse(rxn.gene_reaction_rule, mode="eval"))

    print(f"===== {name} =====")
    print("Reaction:", rxn.id, "|", rxn.name)
    print("Mouse GPR rule:", mouse_rule)
    print(f"{len(mouse)} subunits | {table['value'].isna().sum()} missing | {(table['value'] == 0).sum()} zero")
    print("90th percentile used:", p90)
    print("GPR result (AND = min, OR = sum):", result)
    print("× 50 =", None if result is None else result * 50)
    print("Bounds in model before contsraint and contextualsiton with data:", rxn.bounds)
    display(table.sort_values("value / p90"))

In [9]:
scaled_p90 = (prot_values / p90).clip(upper=1)   # divide by the 90th percentile, then cap anything above 1 at 1
print((scaled_p90 == 1).sum(), "genes capped at 1")
print((scaled_p90 < 1).sum(), "genes below 1")

34 genes capped at 1
297 genes below 1


In [10]:
print(p90)

163.0


Expected: CI_mitoMap (-5.521472392638037, 5.521472392638037)


In [11]:
summary("Complex I")

===== Complex I =====
Reaction: CI_mitoMap | R_CI_mitoMap NADH dehydrogenase
Mouse GPR rule: (ENSMUSG00000020153 and ENSMUSG00000059734 and ENSMUSG00000024099 and ENSMUSG00000005510 and ENSMUSG00000013593 and ENSMUSG00000037916 and ENSMUSG00000025968 and ENSMUSG00000064341 and ENSMUSG00000064345 and ENSMUSG00000064360 and ENSMUSG00000064363 and ENSMUSG00000065947 and ENSMUSG00000064367 and ENSMUSG00000064368 and ENSMUSG00000021606 and ENSMUSG00000020022 and ENSMUSG00000021764 and ENSMUSG00000000399 and ENSMUSG00000030869 and ENSMUSG00000014294 and ENSMUSG00000026032 and ENSMUSG00000023089 and ENSMUSG00000022450 and ENSMUSG00000002379 and (ENSMUSG00000031059 or ENSMUSG00000061633) and ENSMUSG00000028648 and ENSMUSG00000022820 and ENSMUSG00000036199 and ENSMUSG00000033938 and ENSMUSG00000026895 and ENSMUSG00000022354 and ENSMUSG00000040048 and ENSMUSG00000025204 and ENSMUSG00000030647 and ENSMUSG00000002416 and ENSMUSG00000041881 and ENSMUSG00000035674 and ENSMUSG00000027673 and ENSMUSG0

,gene ID,value,value / p90
44,ENSMUSG00000024038,18.0,0.110429
37,ENSMUSG00000035674,24.0,0.147239
20,ENSMUSG00000026032,30.0,0.184049
14,ENSMUSG00000021606,35.0,0.214724
24,ENSMUSG00000031059,40.0,0.245399
18,ENSMUSG00000030869,40.0,0.245399
39,ENSMUSG00000027673,42.0,0.257669
38,ENSMUSG00000029632,42.0,0.257669
45,ENSMUSG00000071014,46.0,0.282209
23,ENSMUSG00000002379,47.0,0.288344


Expected: CII_mitoMap (-10.736196319018406, 10.736196319018406)


In [12]:
summary("Complex II")


===== Complex II =====
Reaction: CII_mitoMap | R_CII_mitoMap succinate dehydrogenase
Mouse GPR rule: (ENSMUSG00000021577 and ENSMUSG00000009863 and ENSMUSG00000058076)
3 subunits | 0 missing | 0 zero
90th percentile used: 163.0
GPR result (AND = min, OR = sum): 0.2147239263803681
× 50 = 10.736196319018406
Bounds in model before contsraint and contextualsiton with data: (-1000.0, 1000.0)


,gene ID,value,value / p90
0,ENSMUSG00000058076,35,0.214724
2,ENSMUSG00000009863,115,0.705521
1,ENSMUSG00000021577,248,1.521472


Expected: CIII_mitoMap (-3.067484662576687, 3.067484662576687)


In [13]:
summary("Complex III")


===== Complex III =====
Reaction: CIII_mitoMap | R_CIII_mitoMap cytochrome c reductase
Mouse GPR rule: (ENSMUSG00000064370 and ENSMUSG00000022551 and ENSMUSG00000063694 and ENSMUSG00000038462 and ENSMUSG00000021520 and ENSMUSG00000063882 and ENSMUSG00000030884 and ENSMUSG00000025651 and ENSMUSG00000020163 and ENSMUSG00000059534 and ENSMUSG00000042298)
11 subunits | 1 missing | 0 zero
90th percentile used: 163.0
GPR result (AND = min, OR = sum): 0.06134969325153374
× 50 = 3.067484662576687
Bounds in model before contsraint and contextualsiton with data: (-1000.0, 1000.0)


,gene ID,value,value / p90
10,ENSMUSG00000020163,10.0,0.061350
0,ENSMUSG00000059534,18.0,0.110429
1,ENSMUSG00000042298,24.0,0.147239
7,ENSMUSG00000063882,39.0,0.239264
4,ENSMUSG00000063694,62.0,0.380368
6,ENSMUSG00000021520,67.0,0.411043
3,ENSMUSG00000022551,74.0,0.453988
5,ENSMUSG00000038462,115.0,0.705521
9,ENSMUSG00000025651,140.0,0.858896
8,ENSMUSG00000030884,173.0,1.061350


Expected: CIV_mitoMap (0, 1.2269938650306749)


In [14]:
summary("Complex IV")


===== Complex IV =====
Reaction: CIV_mitoMap | R_CIV_mitoMap cytochrome c oxidase
Mouse GPR rule: (ENSMUSG00000064351 and ENSMUSG00000064354 and ENSMUSG00000064358 and (ENSMUSG00000031818 or ENSMUSG00000009876) and ENSMUSG00000000088 and ENSMUSG00000061518 and (ENSMUSG00000041697 or ENSMUSG00000030785) and (ENSMUSG00000036751 or ENSMUSG00000051811) and ENSMUSG00000014313 and (ENSMUSG00000074218 or ENSMUSG00000032330 or ENSMUSG00000024248) and ENSMUSG00000031231 and ENSMUSG00000017778 and (ENSMUSG00000035885 or ENSMUSG00000043319))
19 subunits | 5 missing | 0 zero
90th percentile used: 163.0
GPR result (AND = min, OR = sum): 0.024539877300613498
× 50 = 1.2269938650306749
Bounds in model before contsraint and contextualsiton with data: (0.0, 1000.0)


,gene ID,value,value / p90
15,ENSMUSG00000031231,4.0,0.024540
17,ENSMUSG00000035885,4.0,0.024540
2,ENSMUSG00000064358,6.0,0.036810
12,ENSMUSG00000074218,12.0,0.073620
8,ENSMUSG00000030785,18.0,0.110429
13,ENSMUSG00000032330,18.0,0.110429
7,ENSMUSG00000041697,22.0,0.134969
16,ENSMUSG00000017778,32.0,0.196319
14,ENSMUSG00000024248,33.0,0.202454
11,ENSMUSG00000014313,48.0,0.294479


Expected: CV_mitoMap (-0.9202453987730062, 0.9202453987730062)

In [15]:
summary("Complex V")

===== Complex V =====
Reaction: CV_mitoMap | R_CV_mitoMap ATP synthase
Mouse GPR rule: (ENSMUSG00000025428 and ENSMUSG00000025393 and ENSMUSG00000025781 and ENSMUSG00000003072 and ENSMUSG00000016252 and ENSMUSG00000000563 and ENSMUSG00000034566 and ENSMUSG00000050856 and ENSMUSG00000022890 and ENSMUSG00000038690 and ENSMUSG00000038717 and ENSMUSG00000022956 and ENSMUSG00000064357 and ENSMUSG00000064356 and (ENSMUSG00000006057 or ENSMUSG00000062683 or ENSMUSG00000018770))
17 subunits | 3 missing | 0 zero
90th percentile used: 163.0
GPR result (AND = min, OR = sum): 0.018404907975460124
× 50 = 0.9202453987730062
Bounds in model before contsraint and contextualsiton with data: (-1000.0, 1000.0)


,gene ID,value,value / p90
6,ENSMUSG00000006057,1.0,0.006135
7,ENSMUSG00000062683,1.0,0.006135
8,ENSMUSG00000018770,1.0,0.006135
4,ENSMUSG00000016252,18.0,0.110429
12,ENSMUSG00000038690,24.0,0.147239
10,ENSMUSG00000050856,38.0,0.233129
13,ENSMUSG00000038717,40.0,0.245399
3,ENSMUSG00000003072,42.0,0.257669
11,ENSMUSG00000022890,66.0,0.404908
14,ENSMUSG00000022956,84.0,0.515337


_________________________________________________________________________________________________________

## Section 3
### Investigating the ETC Flux Bounds Using the Maximum Value 


In [16]:
print("Maximum:", prot_values.quantile(1.0))

Maximum: 367.0



Using the same methods above, the authors' cardiac notebook (`proteomics_mouse_cardiac_vs_BAT.ipynb`) was run, removing the specified quantile, which defaults normalisation relative to the maximum value (367).

The bounds of the five ETC complexes were investigated. The `summary_max()` cells below recalculate each of these bounds relative to the maximum value, so the "× 50" value printed for each complex should match the bounds below, retrieved from the `proteomics_mouse_cardiac_vs_BAT.ipynb` notebook.

```
CI_mitoMap (-2.452316076294278, 2.452316076294278)
CII_mitoMap (-4.768392370572207, 4.768392370572207)
CIII_mitoMap (-1.3623978201634876, 1.3623978201634876)
CIV_mitoMap (0, 0.544959128065395)
CV_mitoMap (-0.4087193460490463, 0.4087193460490463)
```

Also retrieved from the notebook: 330 constrained reactions, 
           158 unconstrained reactions and
           72 exchange reactions. This matches the number constrained reactions reported in the mitoMammal publication. Therefore it can be concluded that the authors did not specify the E-Flux algorithm to normalise to the 90th percentile.e 
           

In [18]:
max_val = prot_values.max()

def gpr_value_max(node):
    if isinstance(node, ast.Expression):
        return gpr_value_max(node.body)
    if isinstance(node, ast.Name):
        v = vals.get(node.id)
        return None if v is None or pd.isna(v) else v / max_val
    if isinstance(node, ast.BoolOp):
        parts = [p for p in (gpr_value_max(n) for n in node.values) if p is not None]
        if not parts:
            return None
        return min(parts) if isinstance(node.op, ast.And) else sum(parts)

def summary_max(name):
    rid = complexes[name.lower()]
    rxn = model.reactions.get_by_id(rid)

    mouse_rule = re.split(r"\s+or\s+\(?ENSG\d", rxn.gene_reaction_rule)[0]
    mouse = [g.id for g in rxn.genes if g.id.startswith("ENSMUSG")]

    table = pd.DataFrame({"gene ID": mouse})
    table["value"] = table["gene ID"].map(vals)
    table["value / max"] = table["value"] / max_val

    result = gpr_value_max(ast.parse(rxn.gene_reaction_rule, mode="eval"))

    print(f"===== {name} =====")
    print("Reaction:", rxn.id, "|", rxn.name)
    print("Mouse GPR rule:", mouse_rule)
    print(f"{len(mouse)} subunits | {table['value'].isna().sum()} missing | {(table['value'] == 0).sum()} zero")
    print("Maximum used:", max_val)
    print("GPR result (AND = min, OR = sum):", result)
    print("× 50 =", None if result is None else result * 50)
    display(table.sort_values("value / max"))

Expected: CI_mitoMap (-2.452316076294278, 2.452316076294278)


In [19]:
summary_max("Complex I")

===== Complex I =====
Reaction: CI_mitoMap | R_CI_mitoMap NADH dehydrogenase
Mouse GPR rule: (ENSMUSG00000020153 and ENSMUSG00000059734 and ENSMUSG00000024099 and ENSMUSG00000005510 and ENSMUSG00000013593 and ENSMUSG00000037916 and ENSMUSG00000025968 and ENSMUSG00000064341 and ENSMUSG00000064345 and ENSMUSG00000064360 and ENSMUSG00000064363 and ENSMUSG00000065947 and ENSMUSG00000064367 and ENSMUSG00000064368 and ENSMUSG00000021606 and ENSMUSG00000020022 and ENSMUSG00000021764 and ENSMUSG00000000399 and ENSMUSG00000030869 and ENSMUSG00000014294 and ENSMUSG00000026032 and ENSMUSG00000023089 and ENSMUSG00000022450 and ENSMUSG00000002379 and (ENSMUSG00000031059 or ENSMUSG00000061633) and ENSMUSG00000028648 and ENSMUSG00000022820 and ENSMUSG00000036199 and ENSMUSG00000033938 and ENSMUSG00000026895 and ENSMUSG00000022354 and ENSMUSG00000040048 and ENSMUSG00000025204 and ENSMUSG00000030647 and ENSMUSG00000002416 and ENSMUSG00000041881 and ENSMUSG00000035674 and ENSMUSG00000027673 and ENSMUSG0

,gene ID,value,value / max
44,ENSMUSG00000024038,18.0,0.049046
37,ENSMUSG00000035674,24.0,0.065395
20,ENSMUSG00000026032,30.0,0.081744
14,ENSMUSG00000021606,35.0,0.095368
24,ENSMUSG00000031059,40.0,0.108992
18,ENSMUSG00000030869,40.0,0.108992
39,ENSMUSG00000027673,42.0,0.114441
38,ENSMUSG00000029632,42.0,0.114441
45,ENSMUSG00000071014,46.0,0.125341
23,ENSMUSG00000002379,47.0,0.128065


Expected: CII_mitoMap (-4.768392370572207, 4.768392370572207)


In [ ]:
summary_max("Complex II")

===== Complex II =====
Reaction: CII_mitoMap | R_CII_mitoMap succinate dehydrogenase
Mouse GPR rule: (ENSMUSG00000021577 and ENSMUSG00000009863 and ENSMUSG00000058076)
3 subunits | 0 missing | 0 zero
Maximum used: 367
GPR result (AND = min, OR = sum): 0.09536784741144415
× 50 = 4.768392370572207


,gene ID,value,value / max
2,ENSMUSG00000058076,35,0.095368
1,ENSMUSG00000009863,115,0.313351
0,ENSMUSG00000021577,248,0.675749


Expected: CIII_mitoMap (-1.3623978201634876, 1.3623978201634876)

In [ ]:
summary_max("Complex III")

===== Complex III =====
Reaction: CIII_mitoMap | R_CIII_mitoMap cytochrome c reductase
Mouse GPR rule: (ENSMUSG00000064370 and ENSMUSG00000022551 and ENSMUSG00000063694 and ENSMUSG00000038462 and ENSMUSG00000021520 and ENSMUSG00000063882 and ENSMUSG00000030884 and ENSMUSG00000025651 and ENSMUSG00000020163 and ENSMUSG00000059534 and ENSMUSG00000042298)
11 subunits | 1 missing | 0 zero
Maximum used: 367
GPR result (AND = min, OR = sum): 0.027247956403269755
× 50 = 1.3623978201634876


,gene ID,value,value / max
8,ENSMUSG00000020163,10.0,0.027248
9,ENSMUSG00000059534,18.0,0.049046
10,ENSMUSG00000042298,24.0,0.065395
5,ENSMUSG00000063882,39.0,0.106267
2,ENSMUSG00000063694,62.0,0.168937
4,ENSMUSG00000021520,67.0,0.182561
1,ENSMUSG00000022551,74.0,0.201635
3,ENSMUSG00000038462,115.0,0.313351
7,ENSMUSG00000025651,140.0,0.381471
6,ENSMUSG00000030884,173.0,0.471390


Expected: CIV_mitoMap (0, 0.544959128065395)

In [ ]:
summary_max("Complex IV")

===== Complex IV =====
Reaction: CIV_mitoMap | R_CIV_mitoMap cytochrome c oxidase
Mouse GPR rule: (ENSMUSG00000064351 and ENSMUSG00000064354 and ENSMUSG00000064358 and (ENSMUSG00000031818 or ENSMUSG00000009876) and ENSMUSG00000000088 and ENSMUSG00000061518 and (ENSMUSG00000041697 or ENSMUSG00000030785) and (ENSMUSG00000036751 or ENSMUSG00000051811) and ENSMUSG00000014313 and (ENSMUSG00000074218 or ENSMUSG00000032330 or ENSMUSG00000024248) and ENSMUSG00000031231 and ENSMUSG00000017778 and (ENSMUSG00000035885 or ENSMUSG00000043319))
19 subunits | 5 missing | 0 zero
Maximum used: 367
GPR result (AND = min, OR = sum): 0.010899182561307902
× 50 = 0.544959128065395


,gene ID,value,value / max
15,ENSMUSG00000031231,4.0,0.010899
17,ENSMUSG00000035885,4.0,0.010899
2,ENSMUSG00000064358,6.0,0.016349
12,ENSMUSG00000074218,12.0,0.032698
8,ENSMUSG00000030785,18.0,0.049046
13,ENSMUSG00000032330,18.0,0.049046
7,ENSMUSG00000041697,22.0,0.059946
16,ENSMUSG00000017778,32.0,0.087193
14,ENSMUSG00000024248,33.0,0.089918
11,ENSMUSG00000014313,48.0,0.130790


Expected: CV_mitoMap (-0.4087193460490463, 0.4087193460490463)

In [ ]:
summary_max("Complex V")

===== Complex V =====
Reaction: CV_mitoMap | R_CV_mitoMap ATP synthase
Mouse GPR rule: (ENSMUSG00000025428 and ENSMUSG00000025393 and ENSMUSG00000025781 and ENSMUSG00000003072 and ENSMUSG00000016252 and ENSMUSG00000000563 and ENSMUSG00000034566 and ENSMUSG00000050856 and ENSMUSG00000022890 and ENSMUSG00000038690 and ENSMUSG00000038717 and ENSMUSG00000022956 and ENSMUSG00000064357 and ENSMUSG00000064356 and (ENSMUSG00000006057 or ENSMUSG00000062683 or ENSMUSG00000018770))
17 subunits | 3 missing | 0 zero
Maximum used: 367
GPR result (AND = min, OR = sum): 0.008174386920980926
× 50 = 0.4087193460490463


,gene ID,value,value / max
1,ENSMUSG00000062683,1.0,0.002725
2,ENSMUSG00000018770,1.0,0.002725
16,ENSMUSG00000006057,1.0,0.002725
14,ENSMUSG00000016252,18.0,0.049046
5,ENSMUSG00000038690,24.0,0.065395
3,ENSMUSG00000050856,38.0,0.103542
6,ENSMUSG00000038717,40.0,0.108992
13,ENSMUSG00000003072,42.0,0.114441
4,ENSMUSG00000022890,66.0,0.179837
7,ENSMUSG00000022956,84.0,0.228883
